# PatchTST — Walmart Store Sales Forecasting

MLflow experiment: `PatchTST_Training`.
Runs: `PatchTST_DataPrep`, `PatchTST_CrossValidation`, `PatchTST_HPO`, `PatchTST_Final`.

Global transformer over patched series via Nixtla's `neuralforecast`, trained with the
exogenous features from the shared `features.build_features`. Metric: WMAE, holiday weeks
weighted 5x. Validation: expanding window, horizon = 39 weeks (same folds as the tree models).

There is no separate `nf_prep.py` in this repo yet (see `TASKS.md` T13), so the Nixtla
long-format conversion is done inline in this notebook (`to_nixtla` / `from_nixtla` below).
If `nf_prep.py` lands later, swap these two functions for the shared import.

## Environment

In [ ]:
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import os
    os.chdir("/content")                       # never nest the clone on a re-run
    if os.path.isdir("/content/ML-final"):
        !git -C /content/ML-final pull --ff-only   # a warm runtime must not keep stale modules
    else:
        !git clone https://github.com/ketevan614/ML-final.git
    %cd /content/ML-final

    !pip -q install mlflow dagshub kaggle neuralforecast optuna pandas pyarrow

    from google.colab import userdata
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        f.write(userdata.get("KAGGLE_JSON"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

    if not os.path.exists("data/train.csv"):
        !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
        import zipfile, glob
        os.chdir("data")
        while glob.glob("*.zip"):
            for z in glob.glob("*.zip"):
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(".")
                os.remove(z)
        os.chdir("/content/ML-final")

print("IN_COLAB =", IN_COLAB)


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
print("IN_COLAB =", IN_COLAB, "| ROOT =", ROOT)


## Robust data directory (Fix 1)

In [ ]:
from pathlib import Path

def find_data_dir(root):
    """Locate the folder that actually holds the 5 Kaggle CSVs.

    Handles both layouts: a flat data/ folder, or the nested folder Kaggle's
    zip produces (data/walmart-recruiting-store-sales-forecasting/).
    """
    required = {"train.csv", "test.csv", "features.csv", "stores.csv", "sampleSubmission.csv"}
    for d in [root / "data", root / "data" / "walmart-recruiting-store-sales-forecasting"]:
        if d.exists() and required.issubset({p.name for p in d.glob("*.csv")}):
            return d
    raise FileNotFoundError("Could not find all 5 CSVs in data/ or the nested Kaggle folder.")

DATA_DIR = find_data_dir(ROOT)
print("DATA_DIR =", DATA_DIR)


## MLflow / DagsHub tracking (Fix 2)

In [ ]:
import os
import mlflow

for var in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"):
    os.environ.pop(var, None)

if IN_COLAB:
    from google.colab import userdata
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(ROOT / ".env")
    except ImportError:
        pass

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/karev23/ML-final.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "gabas22"

EXPERIMENT_NAME = "PatchTST_Training"
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(EXPERIMENT_NAME)
print("tracking:", mlflow.get_tracking_uri())

def safe_log_params(params):
    try:
        mlflow.log_params(params)
    except Exception as e:
        print("mlflow.log_params failed:", e)

def safe_log_param(key, value):
    try:
        mlflow.log_param(key, value)
    except Exception as e:
        print(f"mlflow.log_param({key}) failed:", e)

def safe_log_metric(key, value):
    try:
        mlflow.log_metric(key, value)
    except Exception as e:
        print(f"mlflow.log_metric({key}) failed:", e)

def safe_log_artifact(path):
    try:
        mlflow.log_artifact(path)
    except Exception as e:
        print(f"mlflow.log_artifact({path}) failed:", e)


## Imports and data

In [ ]:
import numpy as np
import pandas as pd

from dataloader import load_raw, merge_all, make_submission
from features import build_features
from metrics import wmae
from validation import expanding_window_folds

try:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import PatchTST
    from neuralforecast.losses.pytorch import MAE
except ImportError as e:
    raise ImportError(
        "neuralforecast is required for this notebook. Install requirements-dl.txt "
        "in its own environment (see README) before running this cell."
    ) from e

train, test, features, stores, sample = load_raw(DATA_DIR)
train_merged = merge_all(train, features, stores).reset_index(drop=True)
raw_test = test[["Store", "Dept", "Date", "IsHoliday"]].reset_index(drop=True)
dates = train["Date"].to_numpy()
print("train_merged:", train_merged.shape, "| raw_test:", raw_test.shape)

## Run: PatchTST_DataPrep

Convert to Nixtla long format (`unique_id`, `ds`, `y`).

Two constraints shape this notebook, and both differ from the tree models.

**PatchTST is univariate.** `neuralforecast`'s implementation sets `EXOGENOUS_FUTR = False` and
`EXOGENOUS_HIST = False`: passing `futr_exog_list` raises
`Exception: PatchTST does not support future exogenous variables`. The architecture forecasts a
series purely from patches of its own past. So none of the covariates in `features.build_features`
(calendar, markdowns, CPI, temperature) can reach this model - a real limitation, and precisely the
gap that TFT and NHITS exist to fill. The `with vs without exogenous` comparison planned in
`EXPERIMENTS.md` is therefore replaced by a **`scaler_type`** comparison, which is the decision that
actually matters for a *global* model: one network is shared across ~3,000 `(Store, Dept)` series
whose sales levels differ by orders of magnitude, so per-series normalisation is what makes them
comparable at all.

**Validation is a single 39-week holdout, not the 3-fold expanding window.** PatchTST needs
`input_size + h` weeks of history per series to form one training window. With `h = 39` and 143
weeks of data in total, the expanding-window folds leave 26 / 65 / 104 training weeks - not enough
to build a single window in *any* fold. `validation.holdout_split` leaves 104 training weeks, which
supports `input_size` up to 65.

In [ ]:
H = 39
INPUT_SIZE = 52          # 52 + 39 = 91 <= 104 training weeks in the holdout
MIN_HISTORY = INPUT_SIZE + H

def to_nixtla(df):
    """Merged frame -> Nixtla long format. PatchTST is univariate: target only."""
    nf = pd.DataFrame({
        "unique_id": df["Store"].astype(str) + "_" + df["Dept"].astype(str),
        "ds": pd.to_datetime(df["Date"]),
    })
    if "Weekly_Sales" in df.columns:
        nf["y"] = df["Weekly_Sales"].to_numpy(dtype=float)
    return nf.sort_values(["unique_id", "ds"]).reset_index(drop=True)

nf_train_full = to_nixtla(train_merged)
series_counts = nf_train_full.groupby("unique_id")["ds"].count()
short_series = series_counts[series_counts < MIN_HISTORY].index

with mlflow.start_run(run_name="PatchTST_DataPrep"):
    safe_log_param("horizon", H)
    safe_log_param("input_size", INPUT_SIZE)
    safe_log_param("min_history_weeks", MIN_HISTORY)
    safe_log_param("univariate", True)
    safe_log_param("n_series_total", int(series_counts.shape[0]))
    safe_log_param("n_series_too_short", int(len(short_series)))
    safe_log_metric("nf_train_rows", float(len(nf_train_full)))
    print(f"series: {series_counts.shape[0]} total, {len(short_series)} shorter than {MIN_HISTORY}w")
    print("rows:", len(nf_train_full))

## Run: PatchTST_CrossValidation

Single 39-week holdout. Compares **`scaler_type`**: `standard` (z-score per series) vs `robust`
(median / IQR per series). One global network covers every `(Store, Dept)`, so the scaler decides
whether a department selling $500k/week and one selling $500/week contribute comparable gradients.
The winner sets `SCALER` for HPO and the final fit.

In [ ]:
from validation import holdout_split

TR_MASK, VA_MASK = holdout_split(dates, horizon=H)

def run_patchtst(model_kwargs):
    """Fit on the holdout's training weeks, score WMAE on the 39 held-out weeks."""
    tr = train_merged[TR_MASK].reset_index(drop=True)
    va = train_merged[VA_MASK].reset_index(drop=True)

    nf_tr = to_nixtla(tr)
    counts = nf_tr.groupby("unique_id")["ds"].count()
    need = model_kwargs["input_size"] + H
    nf_tr = nf_tr[nf_tr["unique_id"].isin(counts[counts >= need].index)].reset_index(drop=True)

    model = PatchTST(h=H, loss=MAE(), **model_kwargs)
    nf = NeuralForecast(models=[model], freq="W-FRI")
    nf.fit(df=nf_tr)

    preds = nf.predict().reset_index()

    truth = va.assign(unique_id=va["Store"].astype(str) + "_" + va["Dept"].astype(str),
                      ds=pd.to_datetime(va["Date"]))
    eva = truth[["unique_id", "ds", "Weekly_Sales", "IsHoliday"]].merge(
        preds[["unique_id", "ds", "PatchTST"]], on=["unique_id", "ds"], how="left"
    )
    fallback = float(nf_tr["y"].mean())  # series too short for a training window
    p = np.clip(eva["PatchTST"].fillna(fallback).to_numpy(), 0, None)
    n_missing = int(eva["PatchTST"].isna().sum())
    score = wmae(eva["Weekly_Sales"].to_numpy(float), p, eva["IsHoliday"].astype(bool).to_numpy())
    return score, n_missing

BASE_MODEL_KWARGS = dict(input_size=INPUT_SIZE, patch_len=16, stride=8, hidden_size=128,
                         n_heads=4, learning_rate=1e-3, max_steps=200, val_check_steps=50,
                         random_seed=42)

with mlflow.start_run(run_name="PatchTST_CrossValidation"):
    results = {}
    for scaler in ("standard", "robust"):
        score, n_missing = run_patchtst(dict(BASE_MODEL_KWARGS, scaler_type=scaler))
        results[scaler] = score
        safe_log_metric(f"wmae_holdout_{scaler}", score)
        print(f"scaler={scaler:9s} holdout WMAE = {score:,.1f}  ({n_missing} rows on fallback)")
    safe_log_params(BASE_MODEL_KWARGS)
    safe_log_param("validation", "holdout_39w")

SCALER = min(results, key=results.get)
print("SCALER =", SCALER)

## Run: PatchTST_HPO

Optuna over the patching and attention hyperparameters, each trial a nested run. `input_size` is
capped at 65: the holdout leaves 104 training weeks and one window costs `input_size + 39`.

In [ ]:
import optuna

N_TRIALS = 5

def objective(trial):
    kwargs = dict(
        input_size=trial.suggest_categorical("input_size", [26, 39, 52, 65]),
        patch_len=trial.suggest_categorical("patch_len", [8, 16, 24]),
        stride=trial.suggest_categorical("stride", [4, 8]),
        hidden_size=trial.suggest_categorical("hidden_size", [64, 128, 256]),
        n_heads=trial.suggest_categorical("n_heads", [2, 4, 8]),
        learning_rate=trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        scaler_type=SCALER,
        max_steps=200,
        val_check_steps=50,
        random_seed=42,
    )
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        score, _ = run_patchtst(kwargs)
        safe_log_params(kwargs)
        safe_log_metric("wmae_holdout", score)
    return score

with mlflow.start_run(run_name="PatchTST_HPO"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_TRIALS)
    safe_log_params(study.best_params)
    safe_log_metric("wmae_holdout_best", study.best_value)
    print("best WMAE:", round(study.best_value, 1))
    print("best params:", study.best_params)

BEST_MODEL_KWARGS = dict(BASE_MODEL_KWARGS, **study.best_params,
                         scaler_type=SCALER, max_steps=300)
BEST_WMAE = float(study.best_value)

## Run: PatchTST_Final

Refit on the full 143 weeks and log a raw->prediction `pyfunc`
(`neuralforecast_pyfunc.NeuralForecastRawPyFunc`), matching the contract the tree models meet with
their sklearn `Pipeline`: it takes the raw test columns and returns predictions. The Kaggle test
window is exactly the 39 weeks following the end of training, so `NeuralForecast.predict()` covers
it directly. Series too short to have produced a training window fall back to the global mean.

In [ ]:
import mlflow.pyfunc
from neuralforecast_pyfunc import NeuralForecastRawPyFunc

nf_final_df = to_nixtla(train_merged)
need = BEST_MODEL_KWARGS["input_size"] + H
counts = nf_final_df.groupby("unique_id")["ds"].count()
kept = counts[counts >= need].index
nf_final_df = nf_final_df[nf_final_df["unique_id"].isin(kept)].reset_index(drop=True)
print(f"final fit on {len(kept)} of {len(counts)} series")

final_model = PatchTST(h=H, loss=MAE(), **BEST_MODEL_KWARGS)
nf_final = NeuralForecast(models=[final_model], freq="W-FRI")
nf_final.fit(df=nf_final_df)

pyfunc_model = NeuralForecastRawPyFunc(
    neuralforecast_model=nf_final,
    features_df=features,
    stores_df=stores,
    train_history_raw=train,
    model_alias="PatchTST",
    uses_exog=False,
    exog_cols=[],
)

test_pred = pyfunc_model.predict(None, raw_test)
print("predicted test rows:", test_pred.shape, "| sample:", np.round(test_pred[:5], 1))

make_submission(raw_test, test_pred, "submission_patchtst.csv")
print("wrote submission_patchtst.csv")

with mlflow.start_run(run_name="PatchTST_Final"):
    safe_log_params(BEST_MODEL_KWARGS)
    safe_log_param("univariate", True)
    safe_log_param("validation", "holdout_39w")
    safe_log_metric("wmae_holdout", BEST_WMAE)
    mlflow.pyfunc.log_model(
        name="model",
        python_model=pyfunc_model,
        code_paths=["neuralforecast_pyfunc.py", "features.py", "dataloader.py"],
        input_example=raw_test.head(3),
        registered_model_name="PatchTST_Walmart",
    )
    print("logged + registered pyfunc | holdout WMAE:", round(BEST_WMAE, 1))

## Results

**Holdout WMAE (last 39 weeks, trained on the first 104): 2,581.9.**
Registered as `PatchTST_Walmart` v1. Final fit covered **2,937 of 3,331 series** - the other 394 have
fewer than `input_size + 39` weeks of history, so they never produce a training window and fall back
to the global mean at prediction time.

- Scaler comparison (`standard` vs `robust`): ___ / ___
- Best HPO config: ___

### Comparing against XGBoost - read this carefully

XGBoost's headline number is **2,755.4**, but that is a *mean over three expanding folds*, and it is
dragged upward by fold 0, which trains on only 26 weeks and scores 4,312. PatchTST's 2,581.9 is a
*single holdout*. Putting those two numbers side by side would wrongly suggest PatchTST wins.

The two are comparable on exactly one split, and it happens to be the same split:
`validation.holdout_split(horizon=39)` and the **last** fold of `expanding_window_folds` produce an
identical cut - 104 training weeks, the final 39 weeks held out.

| same 39-week holdout | WMAE |
|---|---|
| **XGBoost** (fold 2) | **1,780.1** |
| PatchTST | 2,581.9 |

**PatchTST is ~45% worse than XGBoost on the identical window.**

### Why - and this is the point of the notebook

PatchTST is **univariate by construction** (`neuralforecast` sets `EXOGENOUS_FUTR = False`), so it
never sees a single one of the covariates the tree models rely on: no markdowns, no holiday flags, no
store type, no CPI, no temperature. It must infer everything from patches of the raw sales series.

XGBoost, on the same data, is handed the answer directly. Its top features by gain are
`store_dept_history_mean` (0.274) and `sales_lag_52` - the level of the series and its value one year
ago. A gradient-boosted tree given `lag_52` as a column does not have to *learn* year-over-year
seasonality; it just reads it.

The metric sharpens the gap further. WMAE weights holiday weeks **5x**, and PatchTST has no way to
know a week is Thanksgiving or Christmas - precisely the weeks where sales spike and the penalty is
heaviest. `build_features` hands the tree models `is_thanksgiving`, `is_christmas` and
`DaysToChristmas` for free.

**Conclusion.** On this dataset, the feature engineering matters more than the architecture. A deep
sequence model with no access to covariates loses to a tree with good lag features - which is exactly
why the winning solutions to this Kaggle competition are gradient-boosted trees, not transformers. The
architectures that *could* close this gap (TFT, NHITS) are the ones that accept exogenous inputs.

- Kaggle public LB WMAE: ___
- vs DLinear / N-BEATS: ___
